# Hierarchical Summaries [Step 02.03]

> **MLCourse - Agentic AI - Agent Patterns**

Rolling summarisation has one flat memory that everything is squeezed into. Once
a detail is compressed away, it is gone.

A hierarchy keeps the levels:

```
                    +---------------------+
   LEVEL 2          |   whole-thread gist |   ~60 tokens, always in context
                    +----------+----------+
                               |
             +-----------------+-----------------+
   LEVEL 1   |  chapter A gist |  chapter B gist |   ~90 tokens each
             +--------+--------+--------+--------+
                      |                 |
   LEVEL 0     [ raw turns 1-8 ]  [ raw turns 9-16 ]  ...  kept on disk
```

The gist is always in the window. The detail is retrievable **on demand**. You
zoom in only where the current question needs it.

### What you'll learn

- Building the tree bottom-up, and why bottom-up is the only sane direction.
- **Drill-down**: answering a detail question by fetching one level-0 chunk.
- The measured trade: hierarchy costs more calls to build and far fewer to use.

### Why it matters

Rolling summarisation forces one global answer to "how much detail is worth
keeping?". Real questions vary wildly: "what did we decide overall?" needs the
gist, "what exactly was the pricing objection?" needs the raw turns. A hierarchy
lets you answer both without carrying both.

### Prerequisites

- [02_rolling_summarization](02_rolling_summarization.ipynb)
- [03_rag_advanced](../../../03_rag_advanced) - retrieval, which is what drill-down is.

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


In [2]:
# The shared conversation lives in conversation_data.py next to this notebook,
# because 40 turns pasted at the top of five notebooks would bury the lesson.
from conversation_data import (CONVERSATION, DURABLE_FACTS, EPHEMERAL_MARKERS,
                               PROBE_QUESTIONS, as_text)

print("turns          :", len(CONVERSATION))
print("transcript     : %d approx tokens" % approx_tokens(as_text()))
print("durable facts  :", len(DURABLE_FACTS))
for label, _ in DURABLE_FACTS:
    print("   -", label)

turns          : 42
transcript     : 766 approx tokens
durable facts  : 8
   - product name
   - EU-only hosting
   - budget 12,000 EUR
   - price 49 EUR/shop
   - stack: Postgres + Django
   - no third-party LLM
   - pilot: Radhaus Krueger, March
   - 30-day trial, no free tier


### A grader we will reuse in every notebook of this module


In [ ]:
# The question is never "does the summary read nicely". It is "can the agent
# still answer questions that depend on facts from the start of the thread".

def probe(memory_text: str, label: str, verbose=True):
    """Ask each probe question using ONLY `memory_text` as the agent's memory."""
    SYS = ("You are an assistant continuing a long conversation. The notes below "
           "are ALL you remember of it. Answer from the notes only. If the notes "
           "do not contain the answer, reply exactly: UNKNOWN. Be very brief.")
    hits = []
    for q, expected in PROBE_QUESTIONS:
        out = chat([("system", SYS),
                    ("user", "Your notes:\n%s\n\nQuestion: %s" % (memory_text, q))],
                   temperature=0.0, max_tokens=60)
        a = out.content.strip().lower()
        ok = any(e in a for e in expected)
        hits.append(ok)
        if verbose:
            print("  %s %-52s -> %s" % ("OK  " if ok else "LOST", q[:52],
                                        a.replace("\n", " ")[:60]))
    score = sum(hits) / len(hits)
    print("  %-22s %d/%d  (%.0f%%)  memory size: %d tokens"
          % (label, sum(hits), len(hits), score * 100, approx_tokens(memory_text)))
    return score, hits


### 1. Chunk into chapters

Level 0 is the raw conversation cut into chunks. Two rules:

- **Cut on exchange boundaries**, never inside one.
- **Cut on topic shifts if you can detect them.** We use fixed-size chunks here
  for clarity, but in production a topic-shift detector produces much better
  chapters - a chapter that spans two unrelated topics summarises badly.

In [4]:
CHUNK_TURNS = 8         # 4 exchanges per level-0 chunk

LEVEL0 = [CONVERSATION[i:i + CHUNK_TURNS]
          for i in range(0, len(CONVERSATION), CHUNK_TURNS)]

print("%d level-0 chunks" % len(LEVEL0))
for i, ch in enumerate(LEVEL0):
    print("  chunk %d: %2d turns, %3d tokens | %s..."
          % (i, len(ch), approx_tokens(as_text(ch)), ch[0][1][:46]))

6 level-0 chunks
  chunk 0:  8 turns, 174 tokens | Hi - I'm starting a small SaaS for bike shops ...
  chunk 1:  8 turns, 127 tokens | The product is called Spannerbox, by the way....
  chunk 2:  8 turns, 140 tokens | Right. Let's say 49 EUR per shop per month, bu...
  chunk 3:  8 turns, 143 tokens | How long do you think v1 takes?...
  chunk 4:  8 turns, 146 tokens | Our first pilot customer is Radhaus Krueger in...
  chunk 5:  2 turns,  36 tokens | Let's skip deposits for now. Anything else I'm...


### 2. Build the tree bottom-up

Each level summarises the level below it. Bottom-up matters: a level-1 summary
built from raw turns is grounded; one built from a guess about the whole
conversation is not.

Note that level 1 summarises **summaries**, not raw text. That is a different
prompt - the input is already dense, so the instruction has to be about
*selection*, not *condensation*.

In [5]:
L0_PROMPT = """Summarise this section of a conversation in at most 5 bullet points.

Keep every DECISION, CONSTRAINT, NAME, NUMBER and DATE. Drop small talk and
advice that was not acted on. Bullets only, no preamble.

SECTION:
{text}"""

L1_PROMPT = """You are merging section summaries into one higher-level summary.

The input is already condensed, so do not condense further - SELECT. Keep every
decision, constraint, name, number and date. Drop anything that is only a detail
of how a decision was reached. At most 5 bullets, no preamble.

SECTION SUMMARIES:
{text}"""

L2_PROMPT = """Write the single-paragraph gist of this whole conversation: what it
is about, and the decisions and hard constraints that came out of it. At most 60
words. No preamble."""


def summarise(prompt_template, text, max_tokens=200):
    return chat([("user", prompt_template.format(text=text))],
                temperature=0.0, max_tokens=max_tokens).content.strip()


calls = 0

# --- level 1: one summary per level-0 chunk -----------------------------------
LEVEL1 = []
for i, ch in enumerate(LEVEL0):
    s = summarise(L0_PROMPT, as_text(ch), max_tokens=180)
    LEVEL1.append(s)
    calls += 1
    print("chunk %d -> %d tokens" % (i, approx_tokens(s)))

chunk 0 -> 85 tokens


chunk 1 -> 96 tokens


chunk 2 -> 49 tokens


chunk 3 -> 88 tokens


chunk 4 -> 100 tokens


chunk 5 -> 46 tokens


### level 2: merge pairs of level-1 summaries


In [ ]:
LEVEL2 = []
for i in range(0, len(LEVEL1), 2):
    group = LEVEL1[i:i + 2]
    s = summarise(L1_PROMPT, "\n\n".join(group), max_tokens=180)
    LEVEL2.append(s)
    calls += 1
    print("merged level-1 chunks %s -> %d tokens"
          % (list(range(i, i + len(group))), approx_tokens(s)))

# --- level 3: the whole-thread gist -------------------------------------------
GIST = chat([("user", L2_PROMPT + "\n\n" + "\n\n".join(LEVEL2))],
            temperature=0.0, max_tokens=140).content.strip()
calls += 1

print("\nGIST (%d tokens):\n%s" % (approx_tokens(GIST), GIST))
print("\nLLM calls to build the tree: %d" % calls)


In [7]:
l0_tokens = sum(approx_tokens(as_text(ch)) for ch in LEVEL0)

print("%-12s %8s %10s   %s" % ("level", "nodes", "tokens", "role"))
print("-" * 66)
print("%-12s %8d %10d   %s" % ("level 0 raw", len(LEVEL0), l0_tokens,
                               "on disk, fetched on demand"))
for name, nodes, role in (("level 1", LEVEL1, "chapter summaries"),
                          ("level 2", LEVEL2, "merged summaries"),
                          ("gist", [GIST], "ALWAYS in the context window")):
    print("%-12s %8d %10d   %s"
          % (name, len(nodes), sum(approx_tokens(n) for n in nodes), role))
print("-" * 62)
print("%-12s %8s %10d" % ("full text", "", approx_tokens(as_text())))

level           nodes     tokens   role
------------------------------------------------------------------
level 0 raw         6        766   on disk, fetched on demand
level 1             6        464   chapter summaries
level 2             3        383   merged summaries
gist                1         94   ALWAYS in the context window
--------------------------------------------------------------
full text                    766


### 3. Drill-down: the reason to build a tree

With a flat summary, a detail question either hits or misses. With a tree you can
**route**: read the gist, decide which chapter is relevant, fetch only that
chapter's raw turns.

The retrieval step here is a cheap local embedding lookup, not an LLM call.

In [8]:
import numpy as np
from fastembed import TextEmbedding

_EMB = TextEmbedding("BAAI/bge-small-en-v1.5")


def embed(texts):
    v = np.array(list(_EMB.embed(list(texts))), dtype="float32")
    return v / np.linalg.norm(v, axis=1, keepdims=True)


CHAPTER_VECS = embed(LEVEL1)


def drill_down(question, k=1):
    """Pick the most relevant chapter and return its RAW turns."""
    sims = CHAPTER_VECS @ embed([question])[0]
    best = int(np.argmax(sims))
    return best, float(sims[best]), as_text(LEVEL0[best])


DETAIL_Q = "What exactly did we say about pricing, and what was the reasoning?"
idx, score, raw = drill_down(DETAIL_Q)
print("routed to chunk %d (similarity %.3f)\n" % (idx, score))
print(raw)

routed to chunk 2 (similarity 0.613)

USER: Right. Let's say 49 EUR per shop per month, but I might revisit.
ASSISTANT: Reasonable starting point. Easy to raise later, hard to lower.
USER: Random question - do you know any good bike routes near Freiburg?
ASSISTANT: The Dreisam valley path is lovely and mostly flat.
USER: Ha, thanks. Back to work. What stack would you pick?
ASSISTANT: Postgres, a boring server framework, and server-rendered pages.
USER: We've decided on Postgres and Django. That's locked in.
ASSISTANT: Good - Django's admin alone will save you weeks on the repair board.


### Answer the same detail question three ways and compare cost.


In [ ]:
SYS = ("Answer from the notes provided. If they do not contain the answer, say "
       "UNKNOWN. Two sentences maximum.")


def answer(notes, q):
    out = chat([("system", SYS), ("user", "Notes:\n%s\n\nQuestion: %s" % (notes, q))],
               temperature=0.0, max_tokens=110)
    return out.content.strip(), out.usage_metadata["input_tokens"]


a_gist, t_gist = answer(GIST, DETAIL_Q)
a_drill, t_drill = answer(GIST + "\n\nRelevant excerpt:\n" + raw, DETAIL_Q)
a_full, t_full = answer(as_text(), DETAIL_Q)

print("%-22s %10s  %s" % ("context used", "in tokens", "answer"))
print("-" * 76)
for name, a, t in (("gist only", a_gist, t_gist),
                   ("gist + drilled chunk", a_drill, t_drill),
                   ("entire transcript", a_full, t_full)):
    print("%-22s %10d  %s" % (name, t, a.replace("\n", " ")[:70]))
print()
print("drill-down used %.0f%% of the tokens of the full transcript."
      % (100 * t_drill / t_full))


### Reading that result

The gist alone should say UNKNOWN or something vague - it is 60 words, it cannot
contain the reasoning. The drilled version has the actual exchange. The full
transcript also has it, at several times the token cost.

That is the hierarchy's argument in one table: **same answer, a fraction of the
context**, because you fetched the one chapter that mattered instead of carrying
all of them.

If the drilled answer came out wrong, the usual cause is routing, not
summarisation - the embedding picked the wrong chapter. Check `similarity` and
raise `k` before blaming the model.

### 4. Recall of the always-resident memory

The gist plus chapter summaries is what you would actually keep in the window.
Grade it against the same probes.

In [10]:
RESIDENT = "GIST:\n%s\n\nCHAPTER SUMMARIES:\n%s" % (GIST, "\n\n".join(LEVEL1))
print("hierarchy (gist + chapter summaries) as memory")
h_score, _ = probe(RESIDENT, "hierarchy")

print("\ngist ALONE as memory")
g_score, _ = probe(GIST, "gist only", verbose=False)

hierarchy (gist + chapter summaries) as memory


  OK   What is the product called?                          -> spannerbox


  OK   Where must the product be hosted, and why?           -> it must be hosted in the eu because german bike shop custome


  OK   What is the monthly price per shop?                  -> 49 eur


  OK   Which database and web framework were chosen?        -> postgres and django.


  OK   Who is the first pilot customer and when do they sta -> radhaus krueger, starting in march.


  OK   Is it acceptable to send customer data to a hosted L -> no.
  hierarchy              6/6  (100%)  memory size: 566 tokens

gist ALONE as memory


  gist only              6/6  (100%)  memory size: 94 tokens


In [11]:
print("%-28s %9s %9s" % ("memory", "tokens", "recall"))
print("-" * 50)
for name, text, s in (("gist only", GIST, g_score),
                      ("gist + chapters", RESIDENT, h_score)):
    print("%-28s %9d %8.0f%%" % (name, approx_tokens(text), 100 * s))
print("%-28s %9d %8s" % ("full transcript", approx_tokens(as_text()), "(baseline)"))
print()
print("Build cost: %d LLM calls, paid ONCE." % calls)
print("Query cost: 0 extra LLM calls - drill-down retrieval is a local embedding lookup.")

memory                          tokens    recall
--------------------------------------------------
gist only                           94      100%
gist + chapters                    566      100%
full transcript                    766 (baseline)

Build cost: 10 LLM calls, paid ONCE.
Query cost: 0 extra LLM calls - drill-down retrieval is a local embedding lookup.


### 5. When the hierarchy is worth it

| Use a hierarchy when | Use a flat rolling summary when |
|---|---|
| The thread is long-lived (days, weeks) | The session is one sitting |
| You still have the raw turns in storage | Raw turns are discarded |
| Questions vary between gist-level and detail-level | Questions are all recent-context |
| You can afford a build pass | Every LLM call matters |

The build cost is real: we spent `calls` LLM calls constructing the tree. That is
amortised across every later query, so it is a good trade for a long-lived thread
and a bad one for a ten-turn chat.

> **Note the honest bit:** if `gist + chapters` scored the same as the flat
> rolling summary from notebook 02, the hierarchy has not bought you *recall* -
> it has bought you **drill-down**, which is a different and often more valuable
> thing. Do not oversell it as an accuracy win when what it gives you is optional
> depth.

### 6. Pitfalls

- **Chunking across topics.** A chapter covering two unrelated topics summarises
  into mush and routes badly. Cut on topic shifts where you can.
- **Building top-down.** Summarising from a guess about the whole and then
  filling in detail produces confident fabrication. Always bottom-up.
- **Too many levels.** Three levels is plenty for a conversation. Every extra
  level is another lossy hop.
- **Forgetting the raw turns.** The hierarchy's value is that level 0 still
  exists somewhere. If you delete it, you have built a slower flat summary.
- **Routing on the gist.** Route on chapter summaries, which are specific. The
  gist is too abstract to discriminate between chapters.

### Recap

| Idea | Takeaway |
|---|---|
| Levels, not one summary | Gist resident, detail retrievable |
| Bottom-up | Ground each level in the level below |
| Drill-down | Route to one chapter, fetch its raw turns |
| Build once, query cheaply | The build cost amortises over a long-lived thread |
| Keep level 0 | Delete the raw turns and the hierarchy is pointless |

**Next:** [04_preserve_vs_discard](04_preserve_vs_discard.ipynb) - the decision
underneath all of this: which facts are you not allowed to lose?